In [1]:
import os
import sys
import glob
import logging
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt

from typing import Literal
from pathlib import Path
from instanovo.utils.data_handler import SpectrumDataFrame

from instanovo.transformer.dataset import remove_modifications as clean_peptide

# Fix this later, imports should work without this
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))

[04/11/25 22:48:18] INFO     Enabling RDKit 2024.09.6 jupyter extensions                             ]8;id=6776;file:///home/hjisaac/.cache/pypoetry/virtualenvs/instanovoglyco-u5tn6RZG-py3.10/lib/python3.10/site-packages/rdkit/__init__.py\__init__.py]8;;\:]8;id=554751;file:///home/hjisaac/.cache/pypoetry/virtualenvs/instanovoglyco-u5tn6RZG-py3.10/lib/python3.10/site-packages/rdkit/__init__.py#22\22]8;;\

In [2]:
from common.utils import collect_files, get_or_create_folder, load_ipc_files
from common.logger import get_logger_config
from common.constants import (
    BASE_RAW_DATA_DIR,
    BASE_PROCESSED_DATA_DIR,
    BASE_LOGS_DIR,
    BASE_PLOTS_DIR,
    BASE_REPORTS_CSV_DIR,
)

In [3]:
logger_config = get_logger_config(subdir="scripts")
logging.config.dictConfig(logger_config)
logger = logging.getLogger(__name__)

In [4]:
# Collect each unique_peptide.csv file
peptides_file_paths = [
    path
    for path in collect_files(BASE_REPORTS_CSV_DIR, ext="csv")
    if "unique_peptides" in path
]

assert peptides_file_paths, peptides_file_paths

In [14]:
df = pd.concat([pd.read_csv(file) for file in peptides_file_paths], ignore_index=True)
df.head(20)

,Unique Peptides
0,HNGTGGR
1,SQNCHNSSSR
2,AAGMNHTK
3,ANASHDQPQK
4,HNDSGASECR
5,GGGGGGGGGGGGGSGSSSGSSTSR
6,RQQQQQQQQQQQQK
7,QQQQQQQQQQQQK
8,KNDSGAYR
9,KCLNHTTQK


In [15]:
df["Unique Peptides"].describe()

count                163595
unique                44976
top       AVCMLSNTTAIAEAWAR
freq                     10
Name: Unique Peptides, dtype: object

## Split without Kevin constraint

In [16]:
unique_peptides_df = df["Unique Peptides"].drop_duplicates()

In [18]:
indices = np.arange(len(unique_peptides_df))
np.random.shuffle(indices)
split_ratio = 0.8

split_seperator = int(len(unique_peptides_df) * split_ratio)

# Shuffle the DataFrame indices
shuffled_df = unique_peptides_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Train/test split
train_peptides_df = shuffled_df.iloc[:split_seperator].reset_index(drop=True)
test_peptides_df = shuffled_df.iloc[split_seperator:].reset_index(drop=True)

In [19]:
assert len(train_peptides_df) == 35980, len(train_peptides_df)
assert len(test_peptides_df) == 8996, len(test_peptides_df)

In [20]:
# The zero-copy designs inherited by the SpectrumDataFrame class makes splitting the dataset
# one time complicated. Actually, when we filter an object of the SpectrumDataframe class, the filters
# are kept with the object and are lazily evaluated. So, when an object of the SpectrumDataframe class
# is filtered, a new object of the that class is not returned, but instead it is the old object that
# is mutated. So if, I decide to use the .filter_rows method, I'll have to filter train and test separately
# in different inner contexts. But I guess they should be a way to interact with the predicates held by
# an object of that class.

# v0 => for splitting algorithm without taking into account splitting suggestions from Kevin
# v1 =>


def write_split(
    source_dir: Path | str,
    project_name: Path | str,
    split_name: Literal["train", "valid", "test"],  # noqa
    algorithm_version: Literal["vO", "v1", "v2"],
    potential_peptides_set: set,
    max_charge: int = 10,
):

    file_paths = collect_files(location=source_dir)

    sdf = load_ipc_files(file_paths)
    logger.info(f"Loaded {len(sdf)} entries from {source_dir}")
    sdf = sdf[
        (sdf["precursor_charge"] <= max_charge)
        & (sdf["precursor_charge"] > 0)
        & (sdf["peptide"].apply(lambda x: clean_peptide(x) in potential_peptides_set))
    ]
    sdf["modified_peptide"] = sdf["modified_peptide"].fillna(sdf["peptide"])
    assert (
        sdf["precursor_charge"].between(1, max_charge).all()
    ), "Some precursor_charge values are out of range."
    assert all(
        clean_peptide(p) in potential_peptides_set for p in sdf["peptide"]
    ), "Some peptides are not in the allowed set."
    assert (
        sdf["modified_peptide"].isna().sum() == 0
    ), "Every row should have modified_peptide set"

    logger.info(f"Got {len(sdf)} spectra after filtering by precursor charge")
    logger.info(f"Starting {split_name} split for project {project_name}")
    target_path = BASE_PROCESSED_DATA_DIR / project_name
    filename = f"dataset-ms-glyco_{algorithm_version}_{split_name}.parquet"
    sdf.to_parquet(path=target_path / filename, index=False)
    logger.info(
        f"Saved {len(sdf)} spectra for {split_name} to {target_path} for project {project_name}"
    )

In [21]:
projects_dirs = glob.glob(f"{BASE_RAW_DATA_DIR}/*/")
assert projects_dirs, projects_dirs

In [22]:
# Version 0 for train/test split

dirs_to_ignore = ["PXD044641_PXD035158"]  #
# DOCME: Replace the [] by projects_dirs to make the to script run
for project_dir in []:  # projects_dirs:
    project_name = project_dir.split("/")[-2]
    if project_name in dirs_to_ignore:
        logger.info(f"Skipping project {project_name} as part of projects to ignore")
        continue
    project_file_paths = collect_files(location=project_dir, ext="ipc")

    logger.info(
        f"Collected {len(project_file_paths)} of project {project_name} files from {project_dir}"
    )
    for split_name, peptide_set in [
        ("train", set(train_peptides_df)),
        # ("val", set(val_peptides_df)),
        ("test", set(test_peptides_df)),
    ]:
        write_split(
            project_name=project_name,
            split_name=split_name,
            algorithm_version="v0",
            potential_peptides_set=set(peptide_set),
            source_dir=f"{BASE_RAW_DATA_DIR / project_name}/",
        )

In [23]:
kevin_train_peptides_array = pd.read_csv(
    BASE_REPORTS_CSV_DIR
    / "train_blacklist_overlap_identity_splits_massivekb_from_kevin_1067866_with_glyco_projects_44976_found_15499.csv"
)["Overlapped train peptides"].unique()
kevin_test_peptides_array = pd.read_csv(
    BASE_REPORTS_CSV_DIR
    / "test_overlap_identity_splits_massivekb_from_kevin_33575_with_glyco_projects_44976_found_4136.csv"
)["Overlapped test peptides"].unique()
kevin_val_peptides_array = pd.read_csv(
    BASE_REPORTS_CSV_DIR
    / "valid_overlap_identity_splits_massivekb_from_kevin_13062_with_glyco_projects_44976_found_495.csv"
)["Overlapped valid peptides"].unique()

In [ ]:
# Version 1 for train/test split
logger.info("Starting to split the dataset but taking into account kevin's suggestion")
dirs_to_ignore = ["PXD044641_PXD035158"]  #
# Focus on massivekb

for project_dir in projects_dirs:  # projects_dirs:
    project_name = project_dir.split("/")[-2]

    if project_name in dirs_to_ignore:
        logger.info(f"Skipping project {project_name} as part of projects to ignore")
        continue
    project_file_paths = collect_files(location=project_dir, ext="ipc")

    logger.info(
        f"Collected {len(project_file_paths)} of project {project_name} files from {project_dir}"
    )

    for split_name, kevin_peptide_set in [
        ("train", set(kevin_train_peptides_array)),
        ("valid", set(kevin_val_peptides_array)),
        ("test", set(kevin_test_peptides_array)),
    ]:
        write_split(
            project_name=project_name,
            split_name=split_name,
            algorithm_version="v1",
            potential_peptides_set=set(kevin_peptide_set),
            source_dir=f"{BASE_RAW_DATA_DIR / project_name}/",
        )

2025-04-10 00:41:01,638 - __main__ - INFO - Collected 27 of project PXD026629 files from /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD026629/
2025-04-10 00:41:01,672 - __main__ - INFO - Instantiating SpectrumDataFrame with args=() and kwargs={'source': '/home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD026629/*', 'source_type': 'ipc', 'column_mapping': {'intensity': 'intensity_array', 'mz': 'mz_array'}}
2025-04-10 00:41:01,676 - instanovo.utils.data_handler - INFO - Loading file 001 of 027: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD026629/20180904YLJ-VSV4h-02.ipc
2025-04-10 00:41:01,882 - instanovo.utils.data_handler - INFO - Loading file 002 of 027: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/PXD026629/20180904YLJ-VSV0h-03.ipc
2025-04-10 00:41:02,004 - instanovo.utils.data_handler - INFO - Loading file 003 of 027: /home/hjisaac/AI4Science/instanovo_instadeep/InstanovoGlyco/data/raw/

In [5]:
#
# rr = pd.read_parquet(
#  BASE_PROCESSED_DATA_DIR / "PXD035158/dataset-ms-glyco_v1_train-0001-0001.parquet"
# )
# rr.head()

,index,scan,header,rt,frag_type,collision_energy,precursor_mz,precursor_charge,precursor_intensity,lower_offset,...,peptide_observed_mz,peptide_calc_mz,delta_mass,expectation,hyperscore,nextscore,probability,auc_intensity,protein,experiment_name
0,8949,controllerType=0 controllerNumber=1 scan=8950,FTMS + c NSI d Full ms2 839.3411@hcd33.00 [120...,1294.307551,HCD,33.0,838.839966,2,9.527561e+05,1.0,...,838.8400,838.8351,0.0029,0.149623,8.800,0.000,0.7580,3325199.8,sp|Q8BPN8|DMXL2_MOUSE,Fut8_WT_max_IGP_mousebrain_2.mzML
1,9337,controllerType=0 controllerNumber=1 scan=9338,FTMS + c NSI d Full ms2 1026.7343@hcd33.00 [12...,1350.437945,HCD,33.0,1026.399780,3,1.232893e+06,1.0,...,1026.3997,1026.3933,-0.0021,0.000077,22.207,15.165,0.9932,46660924.0,sp|P97300|NPTN_MOUSE,Fut8_WT_max_IGP_mousebrain_2.mzML
2,9500,controllerType=0 controllerNumber=1 scan=9501,FTMS + c NSI d Full ms2 972.7153@hcd33.00 [120...,1374.236220,HCD,33.0,972.381165,3,3.984886e+06,1.0,...,972.3811,972.3757,-0.0024,0.000124,24.663,13.121,0.9930,78170784.0,sp|P97300|NPTN_MOUSE,Fut8_WT_max_IGP_mousebrain_2.mzML
3,9505,controllerType=0 controllerNumber=1 scan=9506,FTMS + c NSI d Full ms2 1026.7322@hcd33.00 [12...,1375.107693,HCD,33.0,1026.399048,3,1.984562e+06,1.0,...,1026.3990,1026.3933,-0.0025,0.001988,14.423,7.244,0.9566,46660924.0,sp|P97300|NPTN_MOUSE,Fut8_WT_max_IGP_mousebrain_2.mzML
4,9545,controllerType=0 controllerNumber=1 scan=9546,FTMS + c NSI d Full ms2 1108.9292@hcd33.00 [12...,1380.829213,HCD,33.0,1108.929199,2,7.928026e+06,1.0,...,1108.9292,1108.9227,-0.0028,0.000097,25.557,11.495,0.9929,34914600.0,sp|P97300|NPTN_MOUSE,Fut8_WT_max_IGP_mousebrain_2.mzML


In [9]:
# rr["modified_peptide"].head(100)

0                    None
1     N[2191]ASNM[147]EYR
2     N[2029]ASNM[147]EYR
3     N[2191]ASNM[147]EYR
4     N[1330]ASNM[147]EYR
             ...         
95      FGTVPN[1817]GSTER
96      SIAHN[1493]MTTPNK
97          NLN[1330]FSTR
98         KN[1493]STAYFR
99         KN[1493]STAYFR
Name: modified_peptide, Length: 100, dtype: object